### RAG Pipeline - Data Ingestion to VectorDB pipeline  

In [1]:
import os 
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


c:\Users\MAHARSH SHAH\Desktop\YRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
 ### Read all the pdfs inside the directory 
def process_all_pdf(pdf_directory):
    """process all pdfs in the directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    # Find all PDF files recursively 
    pdf_files= list(pdf_dir.glob("**/*.pdf"))

    print(f"Found{len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error:{e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents  

# process all the documents 
all_pdf_documents=process_all_pdf("../data")  

Found5 PDF files to process

Processing:AI Ethics and issues.pdf
Loaded 19 pages

Processing:AWS Cloud computing [Autosaved].pdf
Loaded 23 pages

Processing:Big Data Introduction (1).pdf
Loaded 7 pages

Processing:Flask.pdf
Loaded 15 pages

Processing:steps.pdf
Loaded 2 pages

Total documents loaded: 66


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-08-09T23:26:53+05:30', 'author': 'Ashok HOME', 'moddate': '2023-08-09T23:26:53+05:30', 'source': '..\\data\\pdf\\AI Ethics and issues.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1', 'source_file': 'AI Ethics and issues.pdf', 'file_type': 'pdf'}, page_content='© DataMites®. All Rights Reserved | www.datamites.com                    DATA SCIENCE FOUNDATION\nAI Ethical Issues \nAnd Concerns\n1'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-08-09T23:26:53+05:30', 'author': 'Ashok HOME', 'moddate': '2023-08-09T23:26:53+05:30', 'source': '..\\data\\pdf\\AI Ethics and issues.pdf', 'total_pages': 19, 'page': 1, 'page_label': '2', 'source_file': 'AI Ethics and issues.pdf', 'file_type': 'pdf'}, page_content='© DataMites®. All Rights Reserved | www.datamites.com                    DA

In [4]:
### Text Splitting into chunks 

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs=text_splitter.split_documents(documents)
    print(f"Split{len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk 
    if split_docs:
        print(f"\nExample Chunk")
        print(f"Content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split66 documents into 67 chunks

Example Chunk
Content:© DataMites®. All Rights Reserved | www.datamites.com                    DATA SCIENCE FOUNDATION
AI Ethical Issues 
And Concerns
1...
Metadata:{'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-08-09T23:26:53+05:30', 'author': 'Ashok HOME', 'moddate': '2023-08-09T23:26:53+05:30', 'source': '..\\data\\pdf\\AI Ethics and issues.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1', 'source_file': 'AI Ethics and issues.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-08-09T23:26:53+05:30', 'author': 'Ashok HOME', 'moddate': '2023-08-09T23:26:53+05:30', 'source': '..\\data\\pdf\\AI Ethics and issues.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1', 'source_file': 'AI Ethics and issues.pdf', 'file_type': 'pdf'}, page_content='© DataMites®. All Rights Reserved | www.datamites.com                    DATA SCIENCE FOUNDATION\nAI Ethical Issues \nAnd Concerns\n1'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2023-08-09T23:26:53+05:30', 'author': 'Ashok HOME', 'moddate': '2023-08-09T23:26:53+05:30', 'source': '..\\data\\pdf\\AI Ethics and issues.pdf', 'total_pages': 19, 'page': 1, 'page_label': '2', 'source_file': 'AI Ethics and issues.pdf', 'file_type': 'pdf'}, page_content='© DataMites®. All Rights Reserved | www.datamites.com                    DA

### Embedding and VectorStoreDB 

In [8]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity 

In [20]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("OPENAI_API_KEY"))

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


In [21]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3908.53it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\MAHARSH SHAH\AppData\Local\Temp\ipykernel_14588\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [22]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 134


In [23]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings) 

Generating embeddings for 67 texts...


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.08it/s]

Generated embeddings with shape: (67, 384)
Adding 67 documents to vector store...
Successfully added 67 documents to vector store
Total documents in collection: 201


### Retreiver Pipeline 

In [24]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [25]:
rag_retriever

In [26]:
rag_retriever.retrieve("Hey what is AI ethics ")

Retrieving documents for query: 'Hey what is AI ethics '
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.96it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_f1d7ae50_0',
  'content': '© DataMites®. All Rights Reserved | www.datamites.com                    DATA SCIENCE FOUNDATION\nAI Ethical Issues \nAnd Concerns\n1',
  'metadata': {'moddate': '2023-08-09T23:26:53+05:30',
   'page_label': '1',
   'total_pages': 19,
   'creator': 'Microsoft® PowerPoint® 2013',
   'producer': 'Microsoft® PowerPoint® 2013',
   'content_length': 130,
   'page': 0,
   'author': 'Ashok HOME',
   'file_type': 'pdf',
   'doc_index': 0,
   'creationdate': '2023-08-09T23:26:53+05:30',
   'source_file': 'AI Ethics and issues.pdf',
   'source': '..\\data\\pdf\\AI Ethics and issues.pdf'},
  'similarity_score': 0.39737337827682495,
  'distance': 0.602626621723175,
  'rank': 1},
 {'id': 'doc_69352972_0',
  'content': '© DataMites®. All Rights Reserved | www.datamites.com                    DATA SCIENCE FOUNDATION\nAI Ethical Issues \nAnd Concerns\n1',
  'metadata': {'author': 'Ashok HOME',
   'source_file': 'AI Ethics and issues.pdf',
   'moddate': '2023-08-

In [27]:
rag_retriever.retrieve("What is Flask and Deployment steps")

Retrieving documents for query: 'What is Flask and Deployment steps'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.20it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_782fda9f_61',
  'content': '© DataMites®. All Rights Reserved | www.datamites.com                    DATA SCIENCE FOUNDATION\nFLASK\n• Flask is a open source web framework written in python that helps to develop web applications \neasily.\n• It has two protocols \n1) WSGI [Web server gateway interface]\nSending requests from web server to application server is done \nBy WSGI.\n2) Jinga 2 Template \nIt allows you to send the data from server side to user application  when user makes the request.\n11',
  'metadata': {'source': '..\\data\\pdf\\Flask.pdf',
   'creator': 'Microsoft® PowerPoint® 2013',
   'title': 'MODEL DEPLOYMENT',
   'creationdate': '2023-08-09T23:24:55+05:30',
   'total_pages': 15,
   'doc_index': 61,
   'source_file': 'Flask.pdf',
   'content_length': 465,
   'producer': 'Microsoft® PowerPoint® 2013',
   'page': 10,
   'file_type': 'pdf',
   'page_label': '11',
   'moddate': '2023-08-09T23:24:55+05:30',
   'author': 'Ashok HOME'},
  'similarity_score': 0.18

### Integrating Vector Context Pipeline with LLM output 

In [28]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key='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'

llm= ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)


def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"

    ## generate using the GROQ LLM 
    prompt="""Use the following context to answer the question accurately and concisely.

        Context:
        {context}

        Question: {query}

        Answer:""" 
    
    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [29]:
answer = rag_simple("What is Flask and its deployment steps?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is Flask and its deployment steps?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 25.20it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Flask is an open-source web framework written in Python that helps develop web applications easily. It has two protocols: 

1. WSGI (Web Server Gateway Interface) - sends requests from the web server to the application server.
2. Jinga 2 Template - sends data from the server side to the user application when a request is made.

However, the context does not provide deployment steps for Flask. It only describes what Flask is and its protocols.


### Enhance the RAG Pipeline 

In [30]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is Amazon EC2", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What is Amazon EC2'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.98it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Amazon EC2 (Elastic Compute Cloud) is a cloud computing service provided by Amazon Web Services (AWS) that provides scalable computing capacity in the cloud, allowing users to rent virtual computers to run their own computer applications.
Sources: [{'source': 'AWS Cloud computing [Autosaved].pdf', 'page': 16, 'score': 0.40040791034698486, 'preview': 'AMAZON EC2\nAmazon Elastic Compute Cloud (Amazon EC2) is a cloud computing service provided by Amazon Web Services (AWS) that \nprovides scalable computing capacity in the cloud. EC2 enables users to rent virtual computers on which to run their own \ncomputer applications. With EC2, users can easily la...'}, {'source': 'AWS Cloud computing [Autosaved].pdf', 'page': 16, 'score': 0.40040791034698486, 'preview': 'AMAZON EC2\nAmazon Elastic Compute Cloud (Amazon EC2) is a cloud computing service provided by Amazon Web Services (AWS) that \nprovides scalable computing capacity in the cloud. EC2 enables users to rent virtual computers on

In [32]:
 # --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is cloud computing and what are its types?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is cloud computing and what are its types?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.99it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
WHAT IS CLOUD COMPUTING?
Cloud computing refers to the distribution of computer services such as servers, data storage, databases, networking, 
software, analytics, and intelligence over the internet ("cloud") to provide flexible resources, quicker in

novation, and cost
savings. To put it another way, rather than investing in their own data centres, businesses can rent access to another party's 
infrastructure, such as storage, processing servers, and databases, from a cloud computing service provider and only pay for 
the resources they actually use.
You only pay for the cloud services that you really use, which lowers operational costs, improves infrastructure 
performance, and enables you to grow applications to meet changing business requirements.
3

WHAT IS CLOUD COMPUTING?
Cloud computing refers to the distribution of computer services such as servers, data storage, databases, networking, 
software, analytics, and intelligence over the internet ("cloud") to provide flexible resources, quicker innovation, and cost
savings. To put it another way, rather than investing in their own data centres, businesses can rent access to another party's 
infrastructure, such as storage, processing servers, and databases, from a cloud computin